In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
df = pd.read_csv(ROOT / "data" / "cleaned_tinystories.csv")
docs = df["text_processed"].fillna("").tolist()
TOKEN_PATTERN = r"\S+"
print(len(docs), "stories loaded")


15000 stories loaded


In [2]:
bow = CountVectorizer(token_pattern=TOKEN_PATTERN)
X_bow = bow.fit_transform(docs)
n_cells = X_bow.shape[0] * X_bow.shape[1]
print("Shape:", X_bow.shape)
print(f"Non-zero cells: {X_bow.nnz:,} of {n_cells:,}")
print(f"Sparsity: {1 - X_bow.nnz / n_cells:.2%}")


Shape: (15000, 6298)
Non-zero cells: 1,200,815 of 94,470,000
Sparsity: 98.73%


In [3]:
tfidf = TfidfVectorizer(token_pattern=TOKEN_PATTERN)
X_tfidf = tfidf.fit_transform(docs)
vocab = tfidf.get_feature_names_out()
print("Shape:", X_tfidf.shape)

i = 0
print(df.loc[i, "story_clean"][:150], "...")
weights = pd.Series(X_tfidf[i].toarray()[0], index=vocab)
weights = weights[weights > 0].sort_values(ascending=False)
print("\nHighest weights:\n", weights.head(6).round(3).to_string())
print("\nLowest weights:\n", weights.tail(6).round(3).to_string())


Shape: (15000, 6298)
Once upon a time, there was a big whale swimming in the ocean. The whale was very happy because the water was calm and smooth. Suddenly, a big wave ca ...

Highest weights:
 whale     0.875
the       0.164
boat      0.160
net       0.128
rough     0.121
people    0.114

Lowest weights:
 but      0.013
once     0.013
that     0.013
time     0.013
there    0.012
day      0.011


In [4]:
idf = pd.Series(tfidf.idf_, index=vocab)
print(idf[["the", "be", "a", "and", "whale", "net"]].round(2).to_string())


the      1.01
be       1.00
a        1.00
and      1.00
whale    7.21
net      6.31


In [5]:
rows = []
for low, high in [(1, 1), (2, 2), (3, 3), (1, 3)]:
    vec = CountVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(low, high))
    X = vec.fit_transform(docs)
    rows.append({
        "n-grams": f"({low}, {high})",
        "features": X.shape[1],
        "sparsity %": round(100 * (1 - X.nnz / (X.shape[0] * X.shape[1])), 3),
    })
print(pd.DataFrame(rows).to_string(index=False))


n-grams  features  sparsity %
 (1, 1)      6298      98.729
 (2, 2)    206231      99.928
 (3, 3)    801271      99.980
 (1, 3)   1013800      99.961


In [6]:
toy = [
    "the dog run fast",
    "the cat sleep",
    "the dog and the cat play",
    "a bird sing",
]
# Hand calculation for the word "dog" in document 0
N = len(toy)                                    # 4 documents
tf = toy[0].split().count("dog")                # "dog" appears 1 time in doc 0
df_dog = sum("dog" in d.split() for d in toy)   # "dog" is in 2 documents
idf_dog = np.log((1 + N) / (1 + df_dog)) + 1    # scikit-learn's smoothed idf
print(f"tf = {tf}, df = {df_dog}, idf = ln(5/3) + 1 = {idf_dog:.4f}")
print(f"tf-idf by hand = {tf} x {idf_dog:.4f} = {tf * idf_dog:.4f}")

toy_vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, norm=None)
toy_X = toy_vec.fit_transform(toy)
lib = toy_X[0, toy_vec.vocabulary_["dog"]]
print(f"library value  = {lib:.4f}  match: {np.isclose(lib, tf * idf_dog)}")


tf = 1, df = 2, idf = ln(5/3) + 1 = 1.5108
tf-idf by hand = 1 x 1.5108 = 1.5108
library value  = 1.5108  match: True


In [7]:
words = toy[2].split()
bigrams = [f"{a} {b}" for a, b in zip(words, words[1:])]
print("Bigrams in doc 2 by hand:", bigrams)
print("Count of 'the cat' by hand:", bigrams.count("the cat"))

bi_vec = CountVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(2, 2))
bi_X = bi_vec.fit_transform(toy)
print("Library count:", bi_X[2, bi_vec.vocabulary_["the cat"]])


Bigrams in doc 2 by hand: ['the dog', 'dog and', 'and the', 'the cat', 'cat play']
Count of 'the cat' by hand: 1
Library count: 1
